# LinkedIn Content Automation

This notebook is the learning and prototyping area for the project.

We will use it to understand and test:

- the input data for a LinkedIn post request
- the individual AI agent roles
- structured JSON responses
- the OpenAI API
- the FastAPI microservice

The final application code will later live in the `app` folder.

In [13]:
from pathlib import Path
import sys

print("Python executable:")
print(sys.executable)

print("\nCurrent notebook folder:")
print(Path.cwd())

Python executable:
c:\Users\ChristopherCopony\Projekte\MSAGI-COURSE\assignments\03_linkedin_content_automation\.venv\Scripts\python.exe

Current notebook folder:
c:\Users\ChristopherCopony\Projekte\MSAGI-COURSE\assignments\03_linkedin_content_automation\notebooks


## 1. Input data

The workflow needs structured input before any AI agent can work.

For now, we will describe:

- the company
- the target audience
- the topic
- the preferred tone
- the goal of the post

In [14]:
content_request = {
    "company_name": "FinEdge Mumbai",
    "industry": "Fintech",
    "target_audience": "Finance leaders and fintech professionals",
    "topic": "AI Engineering challenges in fintech and how to overcome them",
    "tone": "Professional, clear, but in a humorous way",
    "goal": "Build thought leadership and encourage discussion",
}

content_request

{'company_name': 'FinEdge Mumbai',
 'industry': 'Fintech',
 'target_audience': 'Finance leaders and fintech professionals',
 'topic': 'AI Engineering challenges in fintech and how to overcome them',
 'tone': 'Professional, clear, but in a humorous way',
 'goal': 'Build thought leadership and encourage discussion'}

## 2. Idea Agent

The Idea Agent receives the content request and creates several possible LinkedIn post ideas.

Later, we will replace the simple logic with an OpenAI call.

In [15]:
def generate_ideas(request): # def creates a reusable function, that takes a request dictionary as input
    topic = request["topic"]
    audience = request["target_audience"]

    ideas = [
        f"Explain why {topic} matters for {audience}.",
        f"Share three practical examples of {topic}.",
        f"Discuss a common misconception about {topic}.",
    ]

    return ideas

In [16]:
post_ideas = generate_ideas(content_request)

In [17]:
post_ideas

['Explain why AI Engineering challenges in fintech and how to overcome them matters for Finance leaders and fintech professionals.',
 'Share three practical examples of AI Engineering challenges in fintech and how to overcome them.',
 'Discuss a common misconception about AI Engineering challenges in fintech and how to overcome them.']

## 3. Draft Agent

The Draft Agent receives:

- the original content request
- one selected post idea

It turns them into a first LinkedIn post draft.

In [18]:
def generate_draft(request, selected_idea):
    company = request["company_name"]
    topic = request["topic"]
    goal = request["goal"]

    draft = (
        f"{selected_idea}\n\n"
        f"At {company}, we believe {topic.lower()} can help financial organizations "
        f"identify suspicious activity earlier and make better-informed decisions.\n\n"
        f"The goal is not to replace human judgment, but to support experts with "
        f"faster and more consistent insights.\n\n"
        f"{goal}"
    )

    return draft

In [19]:
selected_idea = post_ideas[0]

draft_post = generate_draft(content_request, selected_idea)

print(draft_post)

Explain why AI Engineering challenges in fintech and how to overcome them matters for Finance leaders and fintech professionals.

At FinEdge Mumbai, we believe ai engineering challenges in fintech and how to overcome them can help financial organizations identify suspicious activity earlier and make better-informed decisions.

The goal is not to replace human judgment, but to support experts with faster and more consistent insights.

Build thought leadership and encourage discussion


## 4. Hashtag Agent

The Hashtag Agent creates relevant hashtags based on the topic and industry.

In [ ]:
def generate_hashtags(request):
    industry = request["industry"]
    topic = request["topic"]

    hashtags = [
        f"#{industry.replace(' ', '')}",
        "#ArtificialIntelligence",
        "#EnginneringChallenges",
        "#FintechInnovation",
        "#DigitalTransformation",
    ]

    return hashtags

In [21]:
hashtags = generate_hashtags(content_request)

hashtags

['#Fintech',
 '#ArtificialIntelligence',
 '#FraudDetection',
 '#FintechInnovation',
 '#DigitalTransformation']

## 5. Confidence Agent

The Confidence Agent estimates how suitable the draft is.

For now, we use simple rules. Later, the AI model will provide the score.

In [22]:
def calculate_confidence(draft, hashtags):
    score = 0.5

    if len(draft) > 200:
        score += 0.2

    if len(hashtags) >= 3:
        score += 0.2

    if "?" in draft:
        score += 0.1

    return min(score, 1.0)

In [23]:
confidence_score = calculate_confidence(draft_post, hashtags)

confidence_score

0.8999999999999999

## 6. Final structured result

The microservice should later return one structured response containing:

- post ideas
- selected idea
- draft
- confidence score
- hashtags

In [24]:
result = {
    "ideas": post_ideas,
    "selected_idea": selected_idea,
    "draft": draft_post,
    "confidence": confidence_score,
    "hashtags": hashtags,
}

result

{'ideas': ['Explain why AI Engineering challenges in fintech and how to overcome them matters for Finance leaders and fintech professionals.',
  'Share three practical examples of AI Engineering challenges in fintech and how to overcome them.',
  'Discuss a common misconception about AI Engineering challenges in fintech and how to overcome them.'],
 'selected_idea': 'Explain why AI Engineering challenges in fintech and how to overcome them matters for Finance leaders and fintech professionals.',
 'draft': 'Explain why AI Engineering challenges in fintech and how to overcome them matters for Finance leaders and fintech professionals.\n\nAt FinEdge Mumbai, we believe ai engineering challenges in fintech and how to overcome them can help financial organizations identify suspicious activity earlier and make better-informed decisions.\n\nThe goal is not to replace human judgment, but to support experts with faster and more consistent insights.\n\nBuild thought leadership and encourage discu

## 7. JSON output

APIs usually exchange data in JSON format.

JSON looks similar to a Python dictionary, but it is a text format that other systems, such as n8n, can read.

In [25]:
import json

result_json = json.dumps(result, indent=2)

print(result_json)

{
  "ideas": [
    "Explain why AI Engineering challenges in fintech and how to overcome them matters for Finance leaders and fintech professionals.",
    "Share three practical examples of AI Engineering challenges in fintech and how to overcome them.",
    "Discuss a common misconception about AI Engineering challenges in fintech and how to overcome them."
  ],
  "selected_idea": "Explain why AI Engineering challenges in fintech and how to overcome them matters for Finance leaders and fintech professionals.",
  "draft": "Explain why AI Engineering challenges in fintech and how to overcome them matters for Finance leaders and fintech professionals.\n\nAt FinEdge Mumbai, we believe ai engineering challenges in fintech and how to overcome them can help financial organizations identify suspicious activity earlier and make better-informed decisions.\n\nThe goal is not to replace human judgment, but to support experts with faster and more consistent insights.\n\nBuild thought leadership an

## 8. Load the OpenAI API key

The API key is stored in `.env` and loaded without displaying it.

In [26]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)

api_key_loaded = bool(os.getenv("OPENAI_API_KEY")) # bool() converts the value to a boolean, True if the key is present and has a value, False otherwise

print("API key loaded:", api_key_loaded)

API key loaded: True


## 9. Test the OpenAI connection

This cell sends a small request to the OpenAI API and prints the response.

In [27]:
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-5-mini",
    input="Reply with exactly: OpenAI connection works.",
    store=False,
)

print(response.output_text)

OpenAI connection works.


## 10. AI-powered Idea Agent

This version sends the content request to an OpenAI model.

The model receives a specific role and returns three LinkedIn post ideas.

In [28]:
def generate_ai_ideas(request):
    prompt = f"""
Create exactly three different LinkedIn post ideas.

Company: {request["company_name"]}
Industry: {request["industry"]}
Target audience: {request["target_audience"]}
Topic: {request["topic"]}
Tone: {request["tone"]}
Goal: {request["goal"]}

Return one idea per line.
Do not add an introduction.
"""

    response = client.responses.create(
        model="gpt-5-mini",
        instructions=(
            "You are an Idea Agent specializing in professional "
            "LinkedIn content for fintech companies."
        ),
        input=prompt,
        store=False, # store=False means that the response will not be stored in OpenAI's database, which can be useful for privacy or data management reasons
    )

    ideas = [
        line.strip() # Remove leading/trailing whitespace
        for line in response.output_text.splitlines() # splitlines() splits the response text into a list of lines, making it easier to process each idea separately
        if line.strip()
    ]

    return ideas

In [29]:
ai_post_ideas = generate_ai_ideas(content_request)

for idea in ai_post_ideas:
    print(idea)

At FinEdge Mumbai, a "Confessions of an AI Engineer in Finance" thread—start with a wink about teaching a model to "respect the CFO," list 5 hard engineering challenges (data drift, latency SLAs, explainability, auditability, deployment sprawl), offer one-line pragmatic fixes (data contracts, streaming pipelines, model cards, audit logs, MLOps CI/CD) and finish with "Which headache keeps you up at night?"
Mini-case post from FinEdge Mumbai: "We almost shipped an AI that quoted 1998 rates"—funny hook, then a 4-step remediation playbook (shadow testing, canary releases, model SLOs, cross-functional incident runbooks), show before/after impact metrics and invite followers to share their near-miss war stories and lessons learned.
Run a poll: "Which AI engineering issue cost you most this quarter? Data quality / Latency & scale / Explainability & compliance / Deployment & ops"—follow with a witty summary thread that maps top votes to concrete patterns (feature stores, real-time infra, model

## 11. AI-powered Draft Agent

The Draft Agent receives the original request and one selected idea.

It turns them into a complete LinkedIn post.

In [30]:
def generate_ai_draft(request, selected_idea):
    prompt = f"""
Write a LinkedIn post based on the following information.

Selected idea:
{selected_idea}

Company: {request["company_name"]}
Industry: {request["industry"]}
Target audience: {request["target_audience"]}
Topic: {request["topic"]}
Tone: {request["tone"]}
Goal: {request["goal"]}

Requirements:
- Write approximately 120 to 180 words.
- Use short, readable paragraphs.
- Do not include hashtags yet.
- End with a question that encourages discussion.
"""

    response = client.responses.create(
        model="gpt-5-mini",
        instructions=(
            "You are a Draft Agent specializing in professional "
            "LinkedIn posts for fintech companies."
        ),
        input=prompt,
        store=False,
    )

    return response.output_text.strip()

In [31]:
selected_ai_idea = ai_post_ideas[0]

ai_draft_post = generate_ai_draft(
    content_request,
    selected_ai_idea,
)

print(ai_draft_post)

At FinEdge Mumbai we tried teaching a model to "respect the CFO"—it now refuses to approve anything without a cost‑benefit memo 😉.

Five engineering headaches we wrestle with in fintech:
Data drift — enforce data contracts and automated validation.
Latency SLAs — streaming pipelines and edge caching for predictable response times.
Explainability — publish model cards and favor interpretable features.
Auditability — immutable audit logs plus dataset and model versioning.
Deployment sprawl — MLOps CI/CD, infra-as-code and blue/green releases.

None of these are silver bullets, but each is a pragmatic lever you can pull to reduce risk and speed time-to-value.

If you’re building AI in finance, which of these keeps you up at night? Which practical fix helped you sleep better? Which one still haunts your deployments? Which headache keeps you up at night?


## 12. AI-powered Hashtag Agent

The Hashtag Agent creates relevant hashtags for the generated post.

It receives both the original request and the finished draft.

In [32]:
def generate_ai_hashtags(request, draft):
    prompt = f"""
Create exactly five relevant LinkedIn hashtags.

Industry: {request["industry"]}
Topic: {request["topic"]}

LinkedIn post:
{draft}

Requirements:
- Return one hashtag per line.
- Include the # symbol.
- Do not add explanations.
"""

    response = client.responses.create(
        model="gpt-5-mini",
        instructions=(
            "You are a Hashtag Agent specializing in professional "
            "LinkedIn content."
        ),
        input=prompt,
        store=False,
    )

    hashtags = [
        line.strip()
        for line in response.output_text.splitlines()
        if line.strip()
    ]

    return hashtags

In [33]:
ai_hashtags = generate_ai_hashtags(
    content_request,
    ai_draft_post,
)

for hashtag in ai_hashtags:
    print(hashtag)

#FinTech
#AIEngineering
#MLOps
#ModelGovernance
#ExplainableAI


## 13. AI-powered Confidence Agent

The Confidence Agent reviews the draft and estimates how suitable it is for publishing.

It returns:

- a score between `0.0` and `1.0`
- a short explanation

In [34]:
def evaluate_ai_draft(request, draft):
    prompt = f"""
Review this LinkedIn post.

Target audience: {request["target_audience"]}
Tone: {request["tone"]}
Goal: {request["goal"]}

Post:
{draft}

Return only valid JSON in this exact format:

{{
  "confidence": 0.85,
  "reason": "Short explanation"
}}

The confidence must be between 0.0 and 1.0.
"""

    response = client.responses.create(
        model="gpt-5-mini",
        instructions=(
            "You are a quality-control agent for professional LinkedIn content."
        ),
        input=prompt,
        store=False,
    )

    return json.loads(response.output_text)

In [35]:
ai_evaluation = evaluate_ai_draft(
    content_request,
    ai_draft_post,
)

ai_evaluation

{'confidence': 0.92,
 'reason': 'Strong, audience-appropriate post: professional with light humor, clear pain points and pragmatic mitigations, and a good CTA to prompt discussion. Minor fixes: remove the duplicated "keeps you up at night" question, tighten the CTA wording, and consider adding one concrete example or metric to boost credibility.'}

## 14. Final AI result

We now combine the outputs of all agents into one structured result.

This is the same format our FastAPI microservice will later return to n8n.

In [36]:
ai_result = {
    "ideas": ai_post_ideas,
    "selected_idea": selected_ai_idea,
    "draft": ai_draft_post,
    "confidence": ai_evaluation["confidence"],
    "reason": ai_evaluation["reason"],
    "hashtags": ai_hashtags,
}

ai_result

{'ideas': ['At FinEdge Mumbai, a "Confessions of an AI Engineer in Finance" thread—start with a wink about teaching a model to "respect the CFO," list 5 hard engineering challenges (data drift, latency SLAs, explainability, auditability, deployment sprawl), offer one-line pragmatic fixes (data contracts, streaming pipelines, model cards, audit logs, MLOps CI/CD) and finish with "Which headache keeps you up at night?"',
  'Mini-case post from FinEdge Mumbai: "We almost shipped an AI that quoted 1998 rates"—funny hook, then a 4-step remediation playbook (shadow testing, canary releases, model SLOs, cross-functional incident runbooks), show before/after impact metrics and invite followers to share their near-miss war stories and lessons learned.',
  'Run a poll: "Which AI engineering issue cost you most this quarter? Data quality / Latency & scale / Explainability & compliance / Deployment & ops"—follow with a witty summary thread that maps top votes to concrete patterns (feature stores, 

In [37]:
print(json.dumps(ai_result, indent=2))

{
  "ideas": [
    "At FinEdge Mumbai, a \"Confessions of an AI Engineer in Finance\" thread\u2014start with a wink about teaching a model to \"respect the CFO,\" list 5 hard engineering challenges (data drift, latency SLAs, explainability, auditability, deployment sprawl), offer one-line pragmatic fixes (data contracts, streaming pipelines, model cards, audit logs, MLOps CI/CD) and finish with \"Which headache keeps you up at night?\"",
    "Mini-case post from FinEdge Mumbai: \"We almost shipped an AI that quoted 1998 rates\"\u2014funny hook, then a 4-step remediation playbook (shadow testing, canary releases, model SLOs, cross-functional incident runbooks), show before/after impact metrics and invite followers to share their near-miss war stories and lessons learned.",
    "Run a poll: \"Which AI engineering issue cost you most this quarter? Data quality / Latency & scale / Explainability & compliance / Deployment & ops\"\u2014follow with a witty summary thread that maps top votes t